In [1]:
import os
import yaml
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import gc
import json

# ============================================================
#   hourly aggregation 데이터 + MLP 조합
#   - hourly 데이터: duration 300~3600초로 다양 (외삽 문제 해결)
#   - MLP: 외삽 가능 (RF 외삽 한계 없음)
#   - train만 증강 (x2,3,4,6,8,12 배수, 3600초 cap)
#   - 피처: cpu, memory, duration, hour
#   - 타겟: energy_kwh
# ============================================================

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Config 로드
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT     = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
MODEL_PATH     = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH    = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

Device: cuda
GPU: NVIDIA A100-SXM4-80GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# 데이터 로드
# ============================================================
hourly_file = os.path.join(PROCESSED_PATH, "instance_usage_hourly_processed.parquet")
df = pd.read_parquet(hourly_file)
print(f"Hourly rows: {len(df):,}")
print(f"\n[duration 통계]")
print(df['duration'].describe())

Mounted at /content/drive
Hourly rows: 1,454,606

[duration 통계]
count    1.454606e+06
mean     3.265929e+03
std      1.804283e+03
min      1.000000e+00
25%      3.065000e+03
50%      3.600000e+03
75%      3.600000e+03
max      2.353400e+04
Name: duration, dtype: float64


In [5]:
# ============================================================
# 피처 / 타겟
# ============================================================
features = ["cpu", "memory", "duration", "hour"]
target   = "energy_kwh"

X = df[features].values.astype(np.float32)
y = df[target].values.astype(np.float32).reshape(-1, 1)

In [6]:
# ============================================================
# Train / Test 분할 (test는 원본 그대로!)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)
print(f"\nTrain (원본): {len(X_train):,}")
print(f"Test  (원본): {len(X_test):,}")


Train (원본): 1,163,684
Test  (원본): 290,922


In [7]:
# ============================================================
# Train만 증강 (duration x[2,3,4,6,8,12], 3600초 cap)
# duration col index = 2, energy col index = 0 (y)
# ============================================================
MAX_DURATION    = 3600.0
DURATION_IDX    = features.index("duration")   # 2

augmented_X = [X_train]
augmented_y = [y_train]

for multiplier in [2, 3, 4, 6, 8, 12]:
    X_aug = X_train.copy()
    y_aug = y_train.copy()

    new_duration = np.clip(X_aug[:, DURATION_IDX] * multiplier, 0, MAX_DURATION)
    actual_mult  = new_duration / X_aug[:, DURATION_IDX]

    X_aug[:, DURATION_IDX] = new_duration
    y_aug[:, 0]            = y_aug[:, 0] * actual_mult

    augmented_X.append(X_aug)
    augmented_y.append(y_aug)

X_train_aug = np.concatenate(augmented_X, axis=0)
y_train_aug = np.concatenate(augmented_y, axis=0)
del augmented_X, augmented_y; gc.collect()

print(f"Train (증강): {len(X_train_aug):,}")
print(f"Duration range (증강): {X_train_aug[:, DURATION_IDX].min():.0f} ~ {X_train_aug[:, DURATION_IDX].max():.0f} sec")

# ============================================================
# 정규화 (MLP는 스케일링 필수)
# ============================================================
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_sc = scaler_X.fit_transform(X_train_aug)
X_test_sc  = scaler_X.transform(X_test)
y_train_sc = scaler_y.fit_transform(y_train_aug)
y_test_sc  = scaler_y.transform(y_test)

X_train_t = torch.tensor(X_train_sc, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train_sc, dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test_sc,  dtype=torch.float32).to(device)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=1024, shuffle=True
)


Train (증강): 8,145,788
Duration range (증강): 1 ~ 23534 sec


In [8]:
# ============================================================
# MLP 모델 정의 (최적 config: hidden=[512,256,128], drop=0.0)
# ============================================================
class EnergyMLP(nn.Module):
    """
    에너지 예측 MLP
    - 입력: 4개 피처 (cpu, memory, duration, hour)
    - 은닉층: 512 -> 256 -> 128
    - 출력: 1 (energy_kwh)
    """
    def __init__(self, input_size=4, hidden_sizes=[512, 256, 128], dropout_rate=0.0):
        super(EnergyMLP, self).__init__()
        layers = []
        in_size = input_size

        for h in hidden_sizes:
            layers.append(nn.Linear(in_size, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_size = h

        layers.append(nn.Linear(in_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

model = EnergyMLP(input_size=4, hidden_sizes=[512, 256, 128], dropout_rate=0.0).to(device)
print(f"\n[MLP 구조]")
print(model)


[MLP 구조]
EnergyMLP(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [9]:
# ============================================================
# 학습
# ============================================================
EPOCHS    = 30
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
    )

best_loss  = float('inf')
best_state = None

print("\n[학습 시작]")
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(X_batch)
    train_loss /= len(X_train_t)

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_test_t), torch.tensor(y_test_sc, dtype=torch.float32).to(device)).item()

    scheduler.step(val_loss)

    if val_loss < best_loss:
        best_loss  = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 5 == 0:
        print(f"  Epoch {epoch:3d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")


[학습 시작]
  Epoch   5 | Train Loss: 0.000286 | Val Loss: 0.000174
  Epoch  10 | Train Loss: 0.000217 | Val Loss: 0.000153
  Epoch  15 | Train Loss: 0.000169 | Val Loss: 0.000121
  Epoch  20 | Train Loss: 0.000146 | Val Loss: 0.000238
  Epoch  25 | Train Loss: 0.000137 | Val Loss: 0.000138
  Epoch  30 | Train Loss: 0.000130 | Val Loss: 0.000179


In [10]:
# ============================================================
# 평가
# ============================================================
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    y_pred_sc = model(X_test_t).cpu().numpy()

y_pred = scaler_y.inverse_transform(y_pred_sc)
y_true = y_test

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"\n[MLP (hourly + 증강) 성능]")
print(f"  RMSE : {rmse:.8f} kWh")
print(f"  MAE  : {mae:.8f} kWh")
print(f"  R2   : {r2:.8f}")
print(f"  MAPE : {mape:.4f}%")


[MLP (hourly + 증강) 성능]
  RMSE : 0.00046924 kWh
  MAE  : 0.00026482 kWh
  R2   : 0.99997854
  MAPE : 1.3702%


In [11]:
# ============================================================
# 모델 저장
# ============================================================
mlp_file      = os.path.join(MODEL_PATH, config['model']['model_names']['mlp'])
scaler_X_file = os.path.join(MODEL_PATH, config['model']['model_names']['scaler_x'])
scaler_y_file = os.path.join(MODEL_PATH, config['model']['model_names']['scaler_y'])

torch.save(model.state_dict(), mlp_file)
with open(scaler_X_file, 'wb') as f: pickle.dump(scaler_X, f)
with open(scaler_y_file, 'wb') as f: pickle.dump(scaler_y, f)

print(f"\nSaved MLP     : {mlp_file}")
print(f"Saved ScalerX : {scaler_X_file}")
print(f"Saved ScalerY : {scaler_y_file}")

# ============================================================
# 결과 저장
# ============================================================
result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json_mlp'])
with open(result_file, 'r') as f:
    results_json = json.load(f)

results_json['mlp_hourly'] = {
    'method'      : 'MLP (hourly aggregation + train augmentation)',
    'architecture': '4 -> 512 -> 256 -> 128 -> 1',
    'input_size'  : 4,
    'hidden_sizes': [512, 256, 128],
    'epochs'      : EPOCHS,
    'batch_size'  : 1024,
    'lr'          : 1e-4,
    'dropout'     : 0.0,
    'augmentation': {
        'multipliers' : [2, 3, 4, 6, 8, 12],
        'max_duration': MAX_DURATION,
        'train_only'  : True,
    },
    'features'    : features,
    'rmse'        : float(rmse),
    'mae'         : float(mae),
    'r2'          : float(r2),
    'mape'        : float(mape),
}

with open(result_file, 'w') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)

print(f"\nResults updated: {result_file}")
print("Done!")


Saved MLP     : /content/drive/MyDrive/EcoTracing/models/energy_model_mlp.pt
Saved ScalerX : /content/drive/MyDrive/EcoTracing/models/energy_scaler_x.pkl
Saved ScalerY : /content/drive/MyDrive/EcoTracing/models/energy_scaler_y.pkl

Results updated: /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_full_results_mlp.json
Done!
